# Notebook 2 — Reasoning & Output Control

**Topics covered in this notebook:**
4. Chain-of-Thought and Step-Back Prompting
5. Output Formatting and Structured Generation (JSON mode, XML tags, Pydantic)
6. Prompt Sensitivity, Fragility, and Robustness Testing


In [15]:
# ── CELL 0 · Install dependencies (run once) ─────────────────────────────────
import subprocess
result = subprocess.run(
    ["uv", "pip", "install",
     "openai>=3.8.0",
     "pydantic>=2.12.0",
     "python-dotenv>=1.0.0",
     "rich>=14.0.0"],
    capture_output=True, text=True
)
print(result.stdout or "All packages already installed.")

All packages already installed.


In [16]:
import os
import json
from typing import Optional
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from pydantic import BaseModel, Field
from rich import print as rprint
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table

load_dotenv()

# ── Provider detection ───────────────────────────────────────────────────────
# Priority: Azure OpenAI > OpenAI > Groq > Gemini

AZURE_OPENAI_KEY      = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_MODEL    = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")

OPENAI_KEY  = os.getenv("OPENAI_API_KEY", "")
GROQ_KEY    = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY  = os.getenv("GEMINI_API_KEY", "")


# ── 1. Azure OpenAI ──────────────────────────────────────────────────────────
if (
    AZURE_OPENAI_KEY
    and AZURE_OPENAI_ENDPOINT
    and AZURE_OPENAI_MODEL
    and not AZURE_OPENAI_KEY.startswith("...")   # change if you use a placeholder
):
    PROVIDER = "azure_openai"

    # Azure OpenAI v1 endpoint
    endpoint = AZURE_OPENAI_ENDPOINT.rstrip("/")

    client = AzureOpenAI(
        api_key=AZURE_OPENAI_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
    )

    # IMPORTANT:
    # For Azure, MODEL is your Azure deployment name.
    MODEL = AZURE_OPENAI_MODEL


# ── 2. OpenAI ────────────────────────────────────────────────────────────────
elif OPENAI_KEY and not OPENAI_KEY.startswith("sk-..."):
    PROVIDER = "openai"

    client = OpenAI(
        api_key=OPENAI_KEY,
    )

    MODEL = "gpt-4o"


# ── 3. Groq ──────────────────────────────────────────────────────────────────
elif GROQ_KEY and not GROQ_KEY.startswith("gsk_..."):
    PROVIDER = "groq"

    client = OpenAI(
        api_key=GROQ_KEY,
        base_url="https://api.groq.com/openai/v1",
    )

    MODEL = "openai/gpt-oss-120b"


# ── 4. Gemini ────────────────────────────────────────────────────────────────
elif GEMINI_KEY and not GEMINI_KEY.startswith("AIza..."):
    PROVIDER = "gemini"

    client = OpenAI(
        api_key=GEMINI_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    MODEL = "models/gemini-3.7-flash"


# ── No provider ──────────────────────────────────────────────────────────────
else:
    raise EnvironmentError(
        "No valid API key found in .env!\n"
        "Add one of:\n"
        "  AZURE_OPENAI_API_KEY + AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_MODEL\n"
        "  OPENAI_API_KEY\n"
        "  GROQ_API_KEY\n"
        "  GEMINI_API_KEY\n"
        "See .env.example for instructions."
    )


print(f"✓ Provider : {PROVIDER.upper()}")

print(f"✓ Model    : {MODEL}\n")
print(f"✓ Model    : {MODEL}\n")

# ── Shared helper functions ───────────────────────────────────────────────────
def chat(messages: list[dict], model: str = MODEL, temperature: float = 0.3,
         response_format: dict | None = None) -> str:
    kwargs = dict(model=model, messages=messages, temperature=temperature)
    if response_format:
        kwargs["response_format"] = response_format
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


def parse_pydantic(prompt: str, schema: type[BaseModel]) -> BaseModel:
    """
    Parse a structured response into a Pydantic model.

    - Azure OpenAI: uses client.beta.chat.completions.parse() (native SDK support)
    - Groq / Gemini: uses JSON mode + manual Pydantic validation (same result)

    This function abstracts the difference so examples work with all providers.
    """
    if PROVIDER == "azure_openai":
        # Native structured output — SDK validates against schema automatically
        response = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format=schema,
            temperature=0.3,
        )
        return response.choices[0].message.parsed
    else:
        # JSON mode fallback for Groq / Gemini
        # We describe the schema in the prompt and parse manually
        schema_desc = json.dumps(schema.model_json_schema(), indent=2)
        augmented_prompt = (
            f"{prompt}\n\n"
            f"Return ONLY valid JSON matching this schema (no markdown fences, no extra text):\n"
            f"{schema_desc}"
        )
        raw = chat(
            [{"role": "user", "content": augmented_prompt}],
            response_format={"type": "json_object"},
            temperature=0.3,
        )
        return schema.model_validate(json.loads(raw))

def show(content: str, title: str, border: str = "green") -> None:
    rprint(Panel(str(content), title=f"[bold]{title}[/]", border_style=border))

✓ Provider : AZURE_OPENAI
✓ Model    : gpt-4o

✓ Model    : gpt-4o



---
## Part 4 — Chain-of-Thought and Step-Back Prompting

### Why does CoT work?
When a model is forced to write out intermediate steps, it **avoids committing to a wrong answer early**.  
The reasoning tokens act as a scratchpad — the final answer is conditioned on correct intermediate work.

| CoT Variant | How to trigger | Best use case |
|-------------|---------------|---------------|
| **Vanilla CoT** | "Think step by step" | Quick wins on non-reasoning models |
| **Structured CoT** | Explicit `<reasoning>` / `<answer>` tags | Production use, parsing-safe |
| **Step-Back** | Ask a higher-level question first | Complex reasoning requiring principles |
| **Zero-shot CoT** | Append "Let's think step by step" | No examples needed |

### Research insight
> CoT + self-consistency gives the **biggest accuracy boost** for reasoning tasks.  
> Wang et al. (2022) showed **+17.9%** on GSM8K combining CoT + majority voting.  
> **Do NOT use explicit CoT on o-series models** (GPT-o1, o3) — they reason internally; explicit CoT can *hurt*. *(promtable.com, 2026)*  
> Structured CoT improved multi-step reasoning accuracy by **34%** vs direct prompting. *(ibuidl.org, 2026)*

In [17]:
# ── EXAMPLE 4a · Math word problem — Direct vs Vanilla CoT vs Structured CoT ──
problem = """\
A train leaves Chicago at 8:00 AM travelling at 80 mph toward New York (790 miles away).
Another train leaves New York at 9:30 AM travelling at 100 mph toward Chicago.
At what time do the two trains meet? (Answer in AM/PM, Chicago timezone)
"""

# Direct prompting — model often makes arithmetic errors
direct = f"Solve this problem: {problem}"

# Vanilla CoT — just one sentence added
vanilla_cot = f"Solve this problem. Think step by step before giving the final answer.\n\n{problem}"

# Structured CoT — model must fill in labelled sections
structured_cot = f"""\
Solve this problem. Follow the format below exactly:

<reasoning>
Step 1: [Identify all given values]
Step 2: [Set up the equations]
Step 3: [Solve, show all arithmetic]
Step 4: [Convert to clock time]
</reasoning>

<answer>
[Your final answer here — time only]
</answer>

Problem:
{problem}
"""

for label, prompt, border in [
    ("Direct Prompting", direct, "red"),
    ("Vanilla CoT", vanilla_cot, "yellow"),
    ("Structured CoT", structured_cot, "green")
]:
    result = chat([{"role": "user", "content": prompt}], temperature=0.0)
    show(result, label, border)
    print()

╭─────────────────────────────────────────────── Direct Prompting ────────────────────────────────────────────────╮
│ To solve this problem, we need to determine when the two trains meet. Let's break it down step by step.         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 1: Define the variables                                                                                │
│ - Distance between Chicago and New York: **790 miles**                                                          │
│ - Train 1 (Chicago to New York):                                                                                │
│   - Departure time: **8:00 AM**                                                                                 │
│   - Speed: **80 mph**                                                                                           │
│ - Train 2 (New York to Chicago):                                                                                │
│   - Departure time: **9:30 AM**                                                                                 │
│   - Speed: **100 mph**                                                                                          │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 2: Calculate the head start of Train 1                                                                 │
│ Train 1 departs at 8:00 AM, while Train 2 departs at 9:30 AM. This means Train 1 has a **1.5-hour head start**  │
│ before Train 2 begins moving.                                                                                   │
│                                                                                                                 │
│ In 1.5 hours, Train 1 travels:                                                                                  │
│ [                                                                                                               │
│ \text{Distance} = \text{Speed} \times \text{Time} = 80 \, \text{mph} \times 1.5 \, \text{hours} = 120 \,        │
│ \text{miles}.                                                                                                   │
│ \]                                                                                                              │
│                                                                                                                 │
│ So, at 9:30 AM, Train 1 is **120 miles** closer to New York, leaving **790 - 120 = 670 miles** between the two  │
│ trains.                                                                                                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 3: Relative speed of the two trains                                                                    │
│ Once Train 2 starts moving at 9:30 AM, the two trains are moving toward each other. Their **combined speed**    │
│ (relative speed) is:                                                                                            │
│ [                                                     

╭────────────────────────────────────────────────── Vanilla CoT ──────────────────────────────────────────────────╮
│ To solve this problem, let's break it down step by step:                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 1: Define the problem                                                                                  │
│ - Train 1 leaves Chicago at 8:00 AM, traveling at 80 mph toward New York.                                       │
│ - Train 2 leaves New York at 9:30 AM, traveling at 100 mph toward Chicago.                                      │
│ - The distance between Chicago and New York is 790 miles.                                                       │
│ - We need to find the time at which the two trains meet.                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 2: Calculate the head start of Train 1                                                                 │
│ - Train 1 starts at 8:00 AM, while Train 2 starts at 9:30 AM.                                                   │
│ - The time difference between their departures is **1.5 hours**.                                                │
│ - During this time, Train 1 travels:                                                                            │
│   [                                                                                                             │
│   \text{Distance} = \text{Speed} \times \text{Time} = 80 \, \text{mph} \times 1.5 \, \text{hours} = 120 \,      │
│ \text{miles}.                                                                                                   │
│   \]                                                                                                            │
│ - So, at 9:30 AM, Train 1 is **120 miles closer to New York**, leaving **790 - 120 = 670 miles** between the    │
│ two trains.                                                                                                     │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### Step 3: Determine the relative speed of the two trains                                                      │
│ - Train 1 is traveling at 80 mph toward New York.                                                               │
│ - Train 2 is traveling at 100 mph toward Chicago.                                                               │
│ - The **relative speed** of the two trains is:                                                                  │
│   [                                                                                                             │
│   \text{Relative Speed} = 80 \, \text{mph} + 100 \, \text{mph} = 180 \, \text{mph}.                             │
│   \]                                                                                                            │
│                                                                                                                 │
│ ---                                                   

╭──────────────────────────────────────────────── Structured CoT ─────────────────────────────────────────────────╮
│ <reasoning>                                                                                                     │
│ Step 1: Identify all given values                                                                               │
│ - Train 1 (Chicago to New York):                                                                                │
│   Departure time = 8:00 AM                                                                                      │
│   Speed = 80 mph                                                                                                │
│   Distance to New York = 790 miles                                                                              │
│                                                                                                                 │
│ - Train 2 (New York to Chicago):                                                                                │
│   Departure time = 9:30 AM                                                                                      │
│   Speed = 100 mph                                                                                               │
│                                                                                                                 │
│ Step 2: Set up the equations                                                                                    │
│ Let \( t \) represent the time (in hours) after 8:00 AM when the two trains meet.                               │
│ - Distance traveled by Train 1 = \( 80t \)                                                                      │
│ - Distance traveled by Train 2 = \( 100(t - 1.5) \) (since Train 2 starts 1.5 hours later)                      │
│                                                                                                                 │
│ The total distance between Chicago and New York is 790 miles. When the two trains meet, the sum of the          │
│ distances they have traveled will equal 790 miles:                                                              │
│ [                                                                                                               │
│ 80t + 100(t - 1.5) = 790                                                                                        │
│ \]                                                                                                              │
│                                                                                                                 │
│ Step 3: Solve, show all arithmetic                                                                              │
│ Expand the equation:                                                                                            │
│ [                                                                                                               │
│ 80t + 100t - 150 = 790                                                                                          │
│ \]                                                                                                              │
│ Combine like terms:                                                                                             │
│ [                                                                                                               │
│ 180t - 150 = 790                                                                                                │
│ \]                                                                                                              │
│ Add 150 to both sides:                                                                                          │
│ [                                                                                                               │
│ 180t = 940                                            

In [18]:
# ── EXAMPLE 4b · Logic puzzle — structured CoT prevents premature commitment ──
puzzle = """\
Four people — Alice, Bob, Carol, and Dan — each own exactly one pet: 
a cat, a dog, a rabbit, or a fish.

Clues:
1. Alice does not own the fish.
2. Bob owns the dog or the rabbit.
3. Carol owns the cat.
4. Dan does not own the rabbit.

Who owns the fish?
"""

structured_cot_logic = f"""\
Solve the logic puzzle below using the structured reasoning format.

<reasoning>
Step 1 — List all constraints:
Step 2 — Apply constraints one by one, eliminating impossible assignments:
Step 3 — Identify the unique solution:
</reasoning>

<answer>
[Name] owns the fish.
</answer>

Puzzle:
{puzzle}
"""

result = chat([{"role": "user", "content": structured_cot_logic}], temperature=0.0)
show(result, "Logic Puzzle — Structured CoT", "blue")

╭───────────────────────────────────────── Logic Puzzle — Structured CoT ─────────────────────────────────────────╮
│ <reasoning>                                                                                                     │
│ Step 1 — List all constraints:                                                                                  │
│ - There are four people: Alice, Bob, Carol, and Dan.                                                            │
│ - Each person owns exactly one pet: a cat, a dog, a rabbit, or a fish.                                          │
│ - Clue 1: Alice does not own the fish.                                                                          │
│ - Clue 2: Bob owns the dog or the rabbit.                                                                       │
│ - Clue 3: Carol owns the cat.                                                                                   │
│ - Clue 4: Dan does not own the rabbit.                                                                          │
│                                                                                                                 │
│ Step 2 — Apply constraints one by one, eliminating impossible assignments:                                      │
│ - From Clue 3, Carol owns the cat. This means the cat is no longer available for Alice, Bob, or Dan.            │
│ - From Clue 1, Alice does not own the fish. This means Alice must own either the dog or the rabbit.             │
│ - From Clue 2, Bob owns the dog or the rabbit. Since Alice must also own either the dog or the rabbit, Bob and  │
│ Alice must each own one of these two pets.                                                                      │
│ - From Clue 4, Dan does not own the rabbit. Since the cat is already owned by Carol, and the dog or rabbit must │
│ be owned by Alice and Bob, the only pet left for Dan is the fish.                                               │
│                                                                                                                 │
│ Step 3 — Identify the unique solution:                                                                          │
│ - Dan owns the fish.                                                                                            │
│ - Carol owns the cat (from Clue 3).                                                                             │
│ - Bob owns the dog (since Dan owns the fish and Alice cannot own the fish, Alice must own the rabbit, leaving   │
│ the dog for Bob).                                                                                               │
│ - Alice owns the rabbit.                                                                                        │
│                                                                                                                 │
│ </reasoning>                                                                                                    │
│                                                                                                                 │
│ <answer>                                                                                                        │
│ Dan owns the fish.                                                                                              │
│ </answer>                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
# ── EXAMPLE 4c · Step-Back Prompting ────────────────────────────────────────
# Step-Back = ask a GENERAL/ABSTRACT question first,
# then use the answer to ground the specific solution.
# This is especially effective for tasks requiring domain knowledge or principles.

specific_question = """\
Our startup is building a social app. We have 10,000 users and store all their posts 
in a single PostgreSQL table with 2 million rows. Queries are taking 8+ seconds. 
Should we shard the database?
"""

# Direct approach
direct_answer = chat([
    {"role": "user", "content": specific_question}
])

# Step-Back approach: first ask the abstract principle question
step_back_question = """\
What are the general principles and decision criteria for choosing between 
database scaling strategies (vertical scaling, read replicas, sharding, caching, 
indexing)? When is each approach appropriate?
"""
principles = chat([{"role": "user", "content": step_back_question}])

# Now apply principles to the specific case
grounded_answer = chat([
    {"role": "user", "content": step_back_question},
    {"role": "assistant", "content": principles},
    {"role": "user", "content": f"Now apply these principles to our specific situation:\n\n{specific_question}"}
])

show(direct_answer, "Direct Answer (no step-back)", "red")
print()
show(grounded_answer, "Step-Back Answer (principles → specific)", "green")

╭───────────────────────────────────────── Direct Answer (no step-back) ──────────────────────────────────────────╮
│ Sharding the database might eventually be necessary as your app grows, but it is likely overkill at this stage. │
│ Sharding adds significant complexity to your system, so it should generally be considered a last resort after   │
│ you've exhausted other optimization techniques. With 2 million rows, PostgreSQL should still be able to handle  │
│ your workload efficiently if the database is properly optimized. Here are some steps to improve query           │
│ performance before considering sharding:                                                                        │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. **Analyze and Optimize Your Queries**                                                                    │
│    - Use the `EXPLAIN` or `EXPLAIN ANALYZE` command to understand how PostgreSQL executes your queries.         │
│    - Look for expensive operations like sequential scans, unnecessary joins, or inefficient filters.            │
│    - Rewrite queries to minimize complexity and ensure they only fetch the data you need.                       │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 2. **Add Indexes**                                                                                          │
│    - Ensure you have appropriate indexes on the columns used in `WHERE`, `JOIN`, `GROUP BY`, and `ORDER BY`     │
│ clauses.                                                                                                        │
│    - Consider creating composite indexes if your queries filter on multiple columns.                            │
│    - Use **GIN (Generalized Inverted Index)** for full-text search or JSONB fields.                             │
│    - Be cautious about over-indexing, as it can slow down write operations.                                     │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 3. **Vacuum and Analyze**                                                                                   │
│    - Run `VACUUM` and `ANALYZE` regularly to clean up dead tuples and update PostgreSQL's query planner         │
│ statistics.                                                                                                     │
│    - Consider enabling **autovacuum** if it's not already running.                                              │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 4. **Partition the Table**                                                                                  │
│    - If your queries frequently filter by a specific column (e.g., `user_id` or `created_at`), consider **table │
│ partitioning**.                                       

╭─────────────────────────────────── Step-Back Answer (principles → specific) ────────────────────────────────────╮
│ Sharding may not be the best solution for your current situation. Sharding is a complex and advanced scaling    │
│ strategy that is typically reserved for massive datasets or extremely high write throughput that cannot be      │
│ handled by a single database server. With 2 million rows and 10,000 users, your dataset is relatively small by  │
│ modern database standards, and there are simpler, less complex solutions that can significantly improve query   │
│ performance.                                                                                                    │
│                                                                                                                 │
│ Let’s apply the principles and decision criteria to your situation:                                             │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### **1. Understand the Problem**                                                                               │
│ - **Current Dataset Size:** 2 million rows is not an unusually large table for PostgreSQL. Modern relational    │
│ databases can handle tables with tens or hundreds of millions of rows efficiently if properly optimized.        │
│ - **Performance Issue:** Queries are taking 8+ seconds, which suggests inefficiencies in query execution,       │
│ indexing, or database configuration rather than a fundamental need for sharding.                                │
│ - **Growth Trajectory:** With 10,000 users, your current workload is manageable, but you should plan for growth │
│ as your user base expands.                                                                                      │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### **2. Evaluate Simpler Solutions First**                                                                     │
│ Before considering sharding, you should explore simpler and less disruptive strategies to improve query         │
│ performance. These include:                                                                                     │
│                                                                                                                 │
│ #### **a. Indexing**                                                                                            │
│ - **Why:** Indexing is often the first step to optimize query performance. If your queries involve filtering,   │
│ sorting, or joining on specific columns (e.g., `user_id`, `created_at`, `post_id`), adding appropriate indexes  │
│ can drastically reduce query execution time.                                                                    │
│ - **What to Do:**                                                                                               │
│   - Identify slow queries using PostgreSQL's `EXPLAIN` or `EXPLAIN ANALYZE` to see how the query is being       │
│ executed.                                                                                                       │
│   - Add indexes on columns frequently used in `WHERE`, `JOIN`, `ORDER BY`, or `GROUP BY` clauses.               │
│   - Consider composite indexes if queries filter on multiple columns (e.g., `(user_id, created_at)`).           │
│ - **Expected Impact:** Proper indexing can reduce quer

## Part 4 — Chain-of-Thought, Structured CoT, Step-Back, and Reasoning

### What is CoT?
Chain-of-Thought (CoT) prompting asks the model to work through a problem in intermediate steps before producing the final result.

A simple form is:

```text
Solve the problem.
Think step by step before giving the final answer.
```

The important idea is not the exact phrase. The goal is to encourage a more deliberate multi-step solution.

### When CoT is useful
CoT is most useful when:
- The task contains several logical steps.
- Arithmetic or transformations are involved.
- A direct answer tends to produce premature conclusions.
- A non-reasoning model needs additional guidance to perform a multi-step task.

It is not automatically necessary for every prompt.

### Token cost
Reasoning-oriented prompting can consume more tokens because additional reasoning work is generated or performed.

Therefore:
- More reasoning can improve difficult-task performance.
- More reasoning can also increase cost and latency.
- At large scale, unnecessary reasoning can become expensive.

The trade-off is therefore:

**more reasoning → potentially better difficult-task performance → potentially more tokens/cost/latency**

### Vanilla CoT
Vanilla CoT is the simplest form:

```text
Think step by step before giving the final answer.
```

No explicit structure is imposed.

Advantages:
- Very simple.
- Easy to add to an existing prompt.
- Useful as a quick experiment.

Disadvantages:
- The generated response can be verbose.
- The reasoning and final answer are mixed together.
- Extracting one specific value from the response can be inconvenient.

### Structured CoT
Structured CoT explicitly separates the reasoning and answer.

Example:

```xml
<reasoning>
Step 1: Identify the given values.
Step 2: Set up the equation.
Step 3: Perform the calculation.
Step 4: Convert the result into the requested format.
</reasoning>

<answer>
Final answer only.
</answer>
```

The tags are not magic Python syntax. They are instructions that establish a predictable structure in the model's response.

### Why structured CoT is useful
It is particularly useful in pipelines.

Suppose a pipeline needs only:

```text
meeting_time = ...
```

Sending an entire explanation to the next component creates unnecessary processing and parsing.

A structured answer allows the application to isolate the required result.

### Structured CoT for dataset creation
A structured reasoning/answer format can also be useful when creating datasets.

For example:

```xml
<reasoning>
...
</reasoning>

<answer>
...
</answer>
```

This separates:
- the reasoning/training signal
- the final answer

That separation makes the data easier to inspect and process.

### Vanilla vs Structured CoT
| Feature | Vanilla CoT | Structured CoT |
|---|---|---|
| Setup | Very simple | More explicit |
| Reasoning organization | Free-form | Defined sections |
| Human readability | Good | Very good |
| Parsing | More difficult | Easier |
| Pipeline use | Less convenient | More convenient |
| Dataset creation | Possible | Particularly useful |

### Zero-shot CoT
Zero-shot CoT means adding a reasoning instruction without providing worked examples.

Example:

```text
Let's think step by step.
```

The key point is:
- No examples are supplied.
- The instruction itself encourages multi-step reasoning.

Zero-shot approaches can be attractive at large scale because they avoid the token/context cost of including examples.

### Reasoning models
Modern models may have reasoning capabilities internally.

Therefore, explicitly asking a reasoning model to expose a long chain of reasoning is not necessarily the best strategy.

A useful distinction is:

**Non-reasoning model**
→ explicit reasoning prompting may help.

**Reasoning model**
→ internal reasoning may already occur, so focus more on the task, constraints, and desired output.

### Reasoning controls
Some model families expose reasoning/effort controls. These controls can affect how much computation/reasoning is performed.

The important concept is that:
- Prompt wording can influence behavior.
- Model/API reasoning settings can also influence behavior.
- They are separate mechanisms.

### Keywords and token usage
Some model/provider families may assign special behavior to words associated with thinking or reasoning.

Therefore, when working with a particular provider, it is useful to understand whether words such as:
- `think`
- `think harder`
- `ultra think`
- similar reasoning instructions

have special behavior.

Do not assume that a keyword has identical meaning across all models.

### Step-Back prompting
Step-Back prompting introduces an additional abstraction layer.

Instead of immediately answering:

```text
Specific problem → answer
```

the workflow becomes:

```text
Specific problem
      ↓
General principles / higher-level question
      ↓
Apply principles to specific problem
      ↓
Final answer
```

### Example
Specific question:

```text
Our startup has a PostgreSQL table with millions of rows
and queries are slow. Should we shard?
```

Step-Back question:

```text
What are the general principles for choosing between
vertical scaling, read replicas, indexing, caching, and sharding?
```

Then the general principles are passed into a second call that evaluates the actual startup situation.

### Why Step-Back works
It gives the model an opportunity to establish:
- relevant concepts
- decision criteria
- domain principles
- possible alternatives

before making the specific decision.

### When Step-Back is appropriate
Use it when:
- The task is complex.
- The task can be decomposed.
- Domain principles matter.
- Several possible approaches need comparison.

For a trivial question, an additional Step-Back call may simply add unnecessary tokens and latency.

### Step-Back is sequential
The two calls are dependent:

```text
Call 1 → principles
          ↓
Call 2 → principles + specific problem
```

The second call therefore should happen after the first call has produced its result.

This is different from two independent parallel calls.

### CoT and correctness
A major principle:

**A reasoning format does not guarantee a correct answer.**

A model can:
- follow the requested structure,
- produce several reasoning steps,
- and still make a logical or arithmetic mistake.

Therefore, important applications still need:
- validation
- testing
- deterministic calculations where possible
- structured outputs
- application-level checks

### CoT in agents and pipelines
For agentic systems, the most useful output is often not a long explanation.

A downstream component might need only:

```json
{
  "action": "search",
  "query": "..."
}
```

rather than a long reasoning paragraph.

This is why structured outputs become increasingly important as systems become more complex.


In [20]:
# ── EXAMPLE 4b · Logic puzzle — structured CoT prevents premature commitment ──
puzzle = """\
Four people — Alice, Bob, Carol, and Dan — each own exactly one pet: 
a cat, a dog, a rabbit, or a fish.

Clues:
1. Alice does not own the fish.
2. Bob owns the dog or the rabbit.
3. Carol owns the cat.
4. Dan does not own the rabbit.

Who owns the fish?
"""

structured_cot_logic = f"""\
Solve the logic puzzle below using the structured reasoning format.

<reasoning>
Step 1 — List all constraints:
Step 2 — Apply constraints one by one, eliminating impossible assignments:
Step 3 — Identify the unique solution:
</reasoning>

<answer>
[Name] owns the fish.
</answer>

Puzzle:
{puzzle}
"""

result = chat([{"role": "user", "content": structured_cot_logic}], temperature=0.0)
show(result, "Logic Puzzle — Structured CoT", "blue")

╭───────────────────────────────────────── Logic Puzzle — Structured CoT ─────────────────────────────────────────╮
│ <reasoning>                                                                                                     │
│ Step 1 — List all constraints:                                                                                  │
│ - There are four people: Alice, Bob, Carol, and Dan.                                                            │
│ - Each person owns exactly one pet: a cat, a dog, a rabbit, or a fish.                                          │
│ - Clues:                                                                                                        │
│   1. Alice does not own the fish.                                                                               │
│   2. Bob owns the dog or the rabbit.                                                                            │
│   3. Carol owns the cat.                                                                                        │
│   4. Dan does not own the rabbit.                                                                               │
│                                                                                                                 │
│ Step 2 — Apply constraints one by one, eliminating impossible assignments:                                      │
│ - From Clue 3, Carol owns the cat. This means the cat is no longer available for Alice, Bob, or Dan.            │
│ - From Clue 1, Alice does not own the fish. This means Alice must own either the dog or the rabbit.             │
│ - From Clue 2, Bob owns the dog or the rabbit. Since Alice also must own either the dog or the rabbit, Bob and  │
│ Alice cannot both own the same pet. Therefore, one of them owns the dog, and the other owns the rabbit.         │
│ - From Clue 4, Dan does not own the rabbit. Since Carol owns the cat, and Alice and Bob own the dog and rabbit, │
│ Dan must own the fish.                                                                                          │
│                                                                                                                 │
│ Step 3 — Identify the unique solution:                                                                          │
│ - Carol owns the cat.                                                                                           │
│ - Alice and Bob own the dog and rabbit (order depends on further reasoning, but it doesn't affect the fish).    │
│ - Dan owns the fish.                                                                                            │
│                                                                                                                 │
│ </reasoning>                                                                                                    │
│                                                                                                                 │
│ <answer>                                                                                                        │
│ Dan owns the fish.                                                                                              │
│ </answer>                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
# ── EXAMPLE 4c · Step-Back Prompting ────────────────────────────────────────
# Step-Back = ask a GENERAL/ABSTRACT question first,
# then use the answer to ground the specific solution.
# This is especially effective for tasks requiring domain knowledge or principles.

specific_question = """\
Our startup is building a social app. We have 10,000 users and store all their posts 
in a single PostgreSQL table with 2 million rows. Queries are taking 8+ seconds. 
Should we shard the database?
"""

# Direct approach
direct_answer = chat([
    {"role": "user", "content": specific_question}
])

# Step-Back approach: first ask the abstract principle question
step_back_question = """\
What are the general principles and decision criteria for choosing between 
database scaling strategies (vertical scaling, read replicas, sharding, caching, 
indexing)? When is each approach appropriate?
"""
principles = chat([{"role": "user", "content": step_back_question}])

# Now apply principles to the specific case
grounded_answer = chat([
    {"role": "user", "content": step_back_question},
    {"role": "assistant", "content": principles},
    {"role": "user", "content": f"Now apply these principles to our specific situation:\n\n{specific_question}"}
])

show(direct_answer, "Direct Answer (no step-back)", "red")
print()
show(grounded_answer, "Step-Back Answer (principles → specific)", "green")

╭───────────────────────────────────────── Direct Answer (no step-back) ──────────────────────────────────────────╮
│ Sharding the database might eventually be necessary as your application scales, but it is often considered a    │
│ last resort due to its complexity and operational overhead. Before jumping to sharding, there are several       │
│ optimizations you should explore to improve query performance. Many performance issues with PostgreSQL can be   │
│ addressed with proper indexing, query optimization, and database tuning. Here's a step-by-step guide to help    │
│ you diagnose and resolve the problem:                                                                           │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. **Analyze Your Queries**                                                                                 │
│ - Use PostgreSQL's `EXPLAIN` or `EXPLAIN ANALYZE` to understand why your queries are slow.                      │
│   - Identify if there are sequential scans, missing indexes, or inefficient joins.                              │
│   - Look for high-cost operations or table scans that could be avoided.                                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 2. **Add Proper Indexes**                                                                                   │
│ - Ensure you have indexes on columns that are frequently queried, filtered, or used in `WHERE`, `ORDER BY`,     │
│ `GROUP BY`, or `JOIN` clauses.                                                                                  │
│   - For example, if users are querying posts by `user_id`, ensure there is an index on the `user_id` column.    │
│   - Consider composite indexes if queries filter on multiple columns (e.g., `(user_id, created_at)`).           │
│ - Use **GIN indexes** for full-text search if you're searching through post content.                            │
│ - Use **partial indexes** if queries often filter on specific conditions (e.g., `WHERE is_deleted = false`).    │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 3. **Optimize Query Patterns**                                                                              │
│ - Avoid fetching unnecessary data. Use `SELECT` with specific columns instead of `SELECT *`.                    │
│ - Paginate results instead of fetching large datasets at once. Use `LIMIT` and `OFFSET` or, better yet,         │
│ cursor-based pagination for better performance.                                                                 │
│ - Cache frequently accessed or expensive query results using an in-memory store like Redis.                     │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 4. **Vacuum and Analyze**                         

╭─────────────────────────────────── Step-Back Answer (principles → specific) ────────────────────────────────────╮
│ Sharding may not be the best immediate solution for your situation, as sharding introduces significant          │
│ complexity and is typically reserved for scenarios where the dataset or write load exceeds the capacity of a    │
│ single database server. Instead, you should first explore simpler and more cost-effective strategies to         │
│ optimize performance. Let’s analyze your situation step by step and determine the most appropriate approach.    │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### **Key Observations:**                                                                                       │
│ 1. **Current Scale:**                                                                                           │
│    - You have 10,000 users and 2 million rows in a single PostgreSQL table.                                     │
│    - While 2 million rows is a moderately large dataset, it is still manageable for PostgreSQL without sharding │
│ if optimized properly.                                                                                          │
│                                                                                                                 │
│ 2. **Performance Issue:**                                                                                       │
│    - Queries are taking 8+ seconds, which is unacceptable for a social app.                                     │
│    - The performance bottleneck is likely caused by inefficient query execution, lack of indexing, or           │
│ suboptimal schema design.                                                                                       │
│                                                                                                                 │
│ 3. **Workload Characteristics:**                                                                                │
│    - Social apps typically have **read-heavy workloads** (e.g., fetching posts for users) with occasional       │
│ writes (e.g., posting new content).                                                                             │
│    - Queries are likely filtering posts by user ID, timestamp, or other criteria (e.g., fetching recent posts   │
│ for a user or timeline).                                                                                        │
│                                                                                                                 │
│ 4. **Growth Expectations:**                                                                                     │
│    - Your user base and dataset are likely to grow rapidly, so scalability is important.                        │
│    - However, sharding may be premature at this stage given the current scale.                                  │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### **Recommended Approach:**                                                                                   │
│                                                                                                                 │
│ #### **Step 1: Optimize Query Performance**                                                                     │
│ Start by analyzing the slow queries using PostgreSQL's

In [22]:
# ── EXAMPLE 4d · Code debugging with CoT ────────────────────────────────────
buggy_code = """\
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

# Test cases:
print(calculate_average([1, 2, 3, 4, 5]))  # Should print 3.0
print(calculate_average([]))               # What happens here?
print(calculate_average([10]))             # Should print 10.0
"""

debug_prompt = f"""\
Debug the following Python function using structured reasoning.

<reasoning>
Step 1 — What does this function intend to do?
Step 2 — Trace through each test case mentally. What happens?
Step 3 — Identify all bugs (not just the obvious one).
Step 4 — Explain why each bug occurs.
</reasoning>

<fixed_code>
[Corrected function with comments explaining each fix]
</fixed_code>

<summary>
[One-line summary of what was wrong]
</summary>

Code:
```python
{buggy_code}
```
"""

result = chat([{"role": "user", "content": debug_prompt}], temperature=0.0)
show(result, "Code Debugging with Structured CoT", "cyan")

╭────────────────────────────────────── Code Debugging with Structured CoT ───────────────────────────────────────╮
│ <reasoning>                                                                                                     │
│ Step 1 — What does this function intend to do?                                                                  │
│ The function `calculate_average` is designed to compute the average of a list of numbers. It sums up all the    │
│ numbers in the list and divides the total by the number of elements in the list.                                │
│                                                                                                                 │
│ Step 2 — Trace through each test case mentally. What happens?                                                   │
│ - For `calculate_average([1, 2, 3, 4, 5])`:                                                                     │
│   The function calculates the total as `1 + 2 + 3 + 4 + 5 = 15` and divides it by `5`, resulting in `3.0`. This │
│ works as expected.                                                                                              │
│                                                                                                                 │
│ - For `calculate_average([])`:                                                                                  │
│   The function attempts to calculate the total as `0` (since the list is empty) and then divides it by          │
│ `len(numbers)`, which is `0`. This results in a `ZeroDivisionError`.                                            │
│                                                                                                                 │
│ - For `calculate_average([10])`:                                                                                │
│   The function calculates the total as `10` and divides it by `1`, resulting in `10.0`. This works as expected. │
│                                                                                                                 │
│ Step 3 — Identify all bugs (not just the obvious one).                                                          │
│ 1. **Division by zero**: When the input list is empty, the function attempts to divide by `len(numbers)`, which │
│ is `0`. This raises a `ZeroDivisionError`.                                                                      │
│ 2. **No input validation**: The function assumes that the input is always a list of numbers. If the input is    │
│ not a list or contains non-numeric elements, the function will raise a `TypeError` or behave unpredictably.     │
│                                                                                                                 │
│ Step 4 — Explain why each bug occurs.                                                                           │
│ 1. **Division by zero**: The function does not check if the input list is empty before performing the division. │
│ Dividing by zero is undefined in mathematics and raises an error in Python.                                     │
│ 2. **No input validation**: The function does not verify that the input is a list of numbers. If the input is   │
│ invalid (e.g., a string, `None`, or a list with non-numeric elements), the function will fail when trying to    │
│ sum the elements or divide the total.                                                                           │
│                                                                                                                 │
│ </reasoning>                                                                                                    │
│                                                                                                                 │
│ <fixed_code>                                                                                                    │
│ ```python                                             

---
## Part 5 — Output Formatting and Structured Generation

**Why structured output matters:**  
LLM output going into a pipeline (code, DB, API) must be *reliably parseable*.  
Random text formatting breaks downstream systems.

| Method | Reliability | Use when |
|--------|-------------|----------|
| JSON mode (`response_format`) | High | Simple JSON, any model |
| XML tags in prompt | High | Claude-style, readable nesting |
| `client.beta.chat.completions.parse()` | Highest | Pydantic schema enforcement |
| Few-shot format examples | High | When schema not available |

### Research insight
> XML-tagged structured output outperforms JSON-requested output by **11%** on average compliance. *(ibuidl.org, 2026)*  
> Few-shot examples push JSON format compliance from 71% → 94%. *(ibuidl.org, 2026)*

## Part 5 — Output Formatting and Structured Generation

### Why structured output matters
LLM responses often begin as free-form text.

Humans can understand free-form text easily, but software needs predictable structures.

For example, a pipeline may need:

```json
{
  "company": "...",
  "title": "...",
  "salary_min": 250000
}
```

rather than:

```text
The company is ...
The position is ...
The salary appears to be ...
```

Structured generation makes downstream processing easier.

### Typical structured-output approaches

| Method | Main idea | Typical use |
|---|---|---|
| JSON mode | Ask API for JSON | Simple structured data |
| XML tags | Explicit tagged sections | Nested/readable structures |
| Pydantic | Schema + validation | Strong application contracts |
| Few-shot examples | Show desired format | When schema enforcement is unavailable |

### JSON mode
The API can be instructed with:

```python
response_format={"type": "json_object"}
```

Then:

```python
parsed = json.loads(raw)
```

converts the generated JSON text into a Python object.

### Important distinction: syntax vs meaning
A response can be:

```json
{"salary_min": 250000}
```

and still contain the wrong salary.

Therefore:

**Valid JSON ≠ correct information.**

JSON mode helps with syntactic structure. It does not independently verify factual correctness.

### XML tags
XML-style tags provide explicit boundaries:

```xml
<analysis>
    <headline>...</headline>
    <key_facts>
        <fact>...</fact>
        <fact>...</fact>
    </key_facts>
</analysis>
```

The nested structure makes the intended organization clear.

### Regex extraction
Python's `re` module can extract a known tag:

```python
headline = re.search(
    r'<headline>(.*?)</headline>',
    result,
    re.DOTALL
)
```

This is deterministic string processing.

### Limitation of regex
Regex assumes the expected pattern exists.

If the model generates:

```xml
<headline>...</headline>
```

the extraction works.

If it changes the structure unexpectedly, the parser may fail.

Therefore, regex is useful for controlled simple formats, but stronger schema validation is preferable when structure is critical.

### Pydantic
Pydantic provides Python models that describe the expected data.

Example:

```python
class Recipe(BaseModel):
    name: str
    prep_time_minutes: int
    servings: int
```

The schema says:
- `name` should be a string.
- `prep_time_minutes` should be an integer.
- `servings` should be an integer.

### Nested Pydantic models
A model can contain another model:

```python
class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: Optional[str] = None

class Recipe(BaseModel):
    ingredients: list[Ingredient]
```

This creates a nested data structure.

### Field descriptions
Pydantic's:

```python
Field(description="...")
```

adds meaning to a field.

Descriptions are especially useful when schemas are supplied to models or tools because they explain what each field represents.

### Optional fields
For example:

```python
calories_per_serving: Optional[int] = None
```

means that the value may be absent.

### Why Pydantic is powerful
It provides a bridge between:
- LLM-generated data
- Python objects
- application validation
- downstream processing

Instead of manually interpreting arbitrary text, the application works with defined fields.

### Provider differences
The notebook abstracts provider-specific behavior through:

```python
parse_pydantic(...)
```

The OpenAI path uses native SDK parsing.

The fallback path uses JSON mode and then:

```python
schema.model_validate(...)
```

This demonstrates an important general architecture:

```text
LLM
 ↓
Structured response
 ↓
Schema validation
 ↓
Python object
 ↓
Application
```

### Production principle
The more important the output is to the application, the less the application should depend on free-form text.

Move important requirements into:
- schemas
- validation
- application logic
- deterministic code
- explicit constraints.


In [23]:
# ── EXAMPLE 5a · JSON mode — reliable structured data extraction ──────────────
# The response_format={"type": "json_object"} parameter forces valid JSON output.
# The model MUST return parseable JSON — if it can't, it will try to restructure.

job_posting = """\
Senior Machine Learning Engineer — Anthropic

We're looking for an ML Engineer to join our safety research team in San Francisco (hybrid).
You'll work on training runs for large language models and help evaluate model capabilities.

Requirements:
- 5+ years in ML engineering
- Deep experience with PyTorch, JAX, or TensorFlow
- Strong Python skills
- Bonus: experience with RLHF or Constitutional AI

Compensation: $250,000 – $380,000 base + equity
Apply by: March 31, 2025
"""

extract_prompt = f"""\
Extract structured information from this job posting.
Return a JSON object with these exact keys:
- company (string)
- title (string)
- location (string)
- work_mode (string: remote/hybrid/onsite)
- years_experience (integer)
- required_skills (list of strings)
- bonus_skills (list of strings)
- salary_min (integer, in USD, no symbols)
- salary_max (integer, in USD, no symbols)
- application_deadline (string, YYYY-MM-DD format)

Job posting:
{job_posting}
"""

raw = chat(
    [{"role": "user", "content": extract_prompt}],
    response_format={"type": "json_object"},
    temperature=0.0
)

parsed = json.loads(raw)  # This will NEVER fail with json_object mode
rprint(parsed)
print(f"\n✓ Parsed successfully. Salary range: ${parsed['salary_min']:,} – ${parsed['salary_max']:,}")

{
    'company': 'Anthropic',
    'title': 'Senior Machine Learning Engineer',
    'location': 'San Francisco',
    'work_mode': 'hybrid',
    'years_experience': 5,
    'required_skills': ['PyTorch', 'JAX', 'TensorFlow', 'Python'],
    'bonus_skills': ['RLHF', 'Constitutional AI'],
    'salary_min': 250000,
    'salary_max': 380000,
    'application_deadline': '2025-03-31'
}


✓ Parsed successfully. Salary range: $250,000 – $380,000


## Example 5a — JSON Mode

### Goal
Extract information from a job posting into predictable JSON.

### Requested fields
The prompt specifies:
- company
- title
- location
- work mode
- years of experience
- required skills
- bonus skills
- salary minimum
- salary maximum
- application deadline

### API setting

```python
response_format={"type": "json_object"}
```

This requests a JSON object rather than ordinary free-form text.

### Parsing

```python
parsed = json.loads(raw)
```

The generated JSON string becomes a Python dictionary.

Then individual fields can be accessed:

```python
parsed["salary_min"]
parsed["salary_max"]
```

### Why this is better than free text
Without structured output, the application might have to interpret:

```text
The salary is approximately $250K to $380K.
```

With structured output:

```json
{
  "salary_min": 250000,
  "salary_max": 380000
}
```

the application can directly access the numbers.

### Important limitation
JSON mode guarantees/encourages the output format, but it does not independently guarantee that:
- the company is correct,
- the salary is correct,
- the date is correct,
- the extracted information is faithful.

Use validation and evaluation for semantic correctness.

### Rich display
The `rprint()` call is for convenient display.

The actual structured-data operations are:
1. Generate JSON.
2. Parse with `json.loads()`.
3. Access the fields.


In [24]:
# ── EXAMPLE 5b · XML tags — readable, nested, model-friendly ─────────────────
# XML tags are especially effective with Claude models but work well across all.
# They let you nest content clearly and parse with simple string operations.

article = """\
Researchers at MIT have developed a new battery technology that could triple the 
energy density of lithium-ion cells. The breakthrough, published in Nature Energy, 
uses a solid-state electrolyte that eliminates the risk of thermal runaway — 
a major safety concern in current EV batteries. Commercial viability is estimated 
at 5–8 years away, pending scale-up challenges. The team is partnering with 
Toyota and Samsung SDI for pilot production trials.
"""

xml_prompt = f"""\
Analyze the following news article and return your analysis in the XML structure below.
Fill in every tag. Do not add extra tags.

<analysis>
  <headline>One-sentence headline capturing the key development</headline>
  <category>Science/Technology/Business/Politics/Other</category>
  <key_facts>
    <fact>First important fact</fact>
    <fact>Second important fact</fact>
    <fact>Third important fact</fact>
  </key_facts>
  <stakeholders>
    <stakeholder role="who they are">Name</stakeholder>
  </stakeholders>
  <timeline>When this development is expected to matter commercially</timeline>
  <impact_score type="short-term">1-10</impact_score>
  <impact_score type="long-term">1-10</impact_score>
</analysis>

Article:
{article}
"""

result = chat([{"role": "user", "content": xml_prompt}], temperature=0.0)
show(result, "XML-Tagged Structured Analysis", "magenta")

# Demonstrate parsing the XML
import re
headline = re.search(r'<headline>(.*?)</headline>', result, re.DOTALL)
if headline:
    print(f"\n📰 Extracted headline: {headline.group(1).strip()}")

╭──────────────────────────────────────── XML-Tagged Structured Analysis ─────────────────────────────────────────╮
│ ```xml                                                                                                          │
│ <analysis>                                                                                                      │
│   <headline>MIT researchers develop battery technology with triple the energy density of lithium-ion            │
│ cells</headline>                                                                                                │
│   <category>Science</category>                                                                                  │
│   <key_facts>                                                                                                   │
│     <fact>MIT researchers have created a new battery technology with triple the energy density of current       │
│ lithium-ion cells.</fact>                                                                                       │
│     <fact>The technology uses a solid-state electrolyte, addressing safety concerns like thermal                │
│ runaway.</fact>                                                                                                 │
│     <fact>Commercial viability is estimated to be 5–8 years away, with pilot production trials involving Toyota │
│ and Samsung SDI.</fact>                                                                                         │
│   </key_facts>                                                                                                  │
│   <stakeholders>                                                                                                │
│     <stakeholder role="researchers">MIT</stakeholder>                                                           │
│     <stakeholder role="industry partners">Toyota</stakeholder>                                                  │
│     <stakeholder role="industry partners">Samsung SDI</stakeholder>                                             │
│   </stakeholders>                                                                                               │
│   <timeline>5–8 years</timeline>                                                                                │
│   <impact_score type="short-term">4</impact_score>                                                              │
│   <impact_score type="long-term">9</impact_score>                                                               │
│ </analysis>                                                                                                     │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📰 Extracted headline: MIT researchers develop battery technology with triple the energy density of lithium-ion cells
